# Basic transfer learning with cats and dogs data (PyTorch)

> This notebook is a PyTorch port of the original TensorFlow/Keras lab. Keras' `ImageDataGenerator` + `flow_from_directory` become torchvision `transforms` + `ImageFolder`, `InceptionV3` comes from `torchvision.models`, and `model.fit` becomes an explicit training loop.

### Tensor shapes in PyTorch — quick reference

- **Channels-first**: images are `(C, H, W)`, not Keras' `(H, W, C)`. `ToTensor()` converts for you.
- **Batch dimension**: layers expect a leading `N`. `DataLoader` turns `(C, H, W)` samples into `(N, C, H, W)` batches. A single image needs it added back with `.unsqueeze(0)`.
- **Conv layers**: shrink `H, W`, grow `C`. This notebook finds the exact output shape empirically (a dummy `torch.zeros(1, 3, 150, 150)` forward pass) instead of computing it by hand.
- **Flatten + Linear**: `nn.Flatten()` maps `(N, C, H, W)` → `(N, C*H*W)`. The next `Linear` layer's `in_features` must equal `C*H*W`, computed here as `np.prod(shape[1:])` (all dims except the batch dim).

Look for `# shape: (...)` comments below to track tensors through the code.


### Import modules and download the cats and dogs dataset.

In [1]:
# --- stdlib ---
import urllib.request        # download dataset
import os                    # paths, dirs, listing
import zipfile                # unzip dataset
import random                 # shuffle file list
from shutil import copyfile   # copy files into split folders

# --- PyTorch / torchvision ---
import numpy as np
import torch
from torch import nn                                               # layers, losses
from torch.utils.data import DataLoader                            # batches a Dataset
from torchvision import transforms                                 # preprocessing/augmentation
from torchvision.datasets import ImageFolder                       # loads class_name/xxx.jpg folders
from torchvision.models import inception_v3, Inception_V3_Weights  # pretrained backbone
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate a few truncated dataset images

# pick an accelerator: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# download + unzip dataset (both steps skipped if already done)
data_url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
data_file_name = "data/kagglecatsanddogs_5340.zip"
download_dir = 'data/catsdogs/'
os.makedirs("data", exist_ok=True)
if not os.path.exists(data_file_name):
    urllib.request.urlretrieve(data_url, data_file_name)
if not os.path.exists(os.path.join(download_dir, 'PetImages')):
    zip_ref = zipfile.ZipFile(data_file_name, 'r')
    zip_ref.extractall(download_dir)
    zip_ref.close()


Using device: mps


In [2]:
# path to the downloaded zip file
data_file_name


'data/kagglecatsanddogs_5340.zip'

Check that the dataset has the expected number of examples.

In [3]:
# count files in each raw class folder
print("Number of cat images:",len(os.listdir('data/catsdogs/PetImages/Cat/')))
print("Number of dog images:", len(os.listdir('data/catsdogs/PetImages/Dog/')))

# Expected Output:
# Number of cat images: 12501
# Number of dog images: 12501


Number of cat images: 12501
Number of dog images: 12501


Create some folders that will store the training and test data.
- There will be a training folder and a testing folder.
- Each of these will have a subfolder for cats and another subfolder for dogs.

In [4]:
# folder layout ImageFolder expects: <split>/<class>/*.jpg
# no-op (via except) if the folders already exist
try:
    os.mkdir('data/cats-v-dogs')
    os.mkdir('data/cats-v-dogs/training')
    os.mkdir('data/cats-v-dogs/testing')
    os.mkdir('data/cats-v-dogs/training/cats')
    os.mkdir('data/cats-v-dogs/training/dogs')
    os.mkdir('data/cats-v-dogs/testing/cats')
    os.mkdir('data/cats-v-dogs/testing/dogs')
except OSError:
    pass


### Split data into training and test sets

- The following code first checks if an image file is empty (zero length) or cannot be decoded as an image (Keras' generator silently skipped such files; `ImageFolder` would raise an error, so we filter them here instead).
- Of the files that are valid, it puts 90% of the data into the training set, and 10% into the test set.

In [5]:
def is_valid_image(path):
    '''
    Checks that a file is a real, non-empty image.

    Args:
      path (string) -- path to the candidate file

    Returns:
      bool -- True if the file is non-empty and PIL can open it
    '''
    if os.path.getsize(path) == 0:
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def split_data(SOURCE, TRAINING, TESTING, SPLIT_SIZE):
    '''
    Splits the images in SOURCE into a training folder and a testing folder.

    Files that are empty or cannot be decoded as images are skipped. The rest are
    shuffled, then SPLIT_SIZE of them are copied to TRAINING and the remainder to
    TESTING. Both destinations are emptied first, so re-running this is safe and
    will not stack a second split on top of the first.

    Args:
      SOURCE (string) -- folder holding the raw images of one class
      TRAINING (string) -- destination folder for the training split
      TESTING (string) -- destination folder for the testing split
      SPLIT_SIZE (float) -- fraction of images sent to TRAINING (0.9 keeps 90%)
    '''
    # keep only valid image files
    files = []
    for filename in os.listdir(SOURCE):
        file = SOURCE + filename
        if is_valid_image(file):
            files.append(filename)
        else:
            print(filename + " is zero length or not a valid image, so ignoring.")

    # shuffle, then split by SPLIT_SIZE fraction
    training_length = int(len(files) * SPLIT_SIZE)
    testing_length = int(len(files) - training_length)
    shuffled_set = random.sample(files, len(files))
    training_set = shuffled_set[0:training_length]
    testing_set = shuffled_set[training_length:]

    # clear the destination folders first, so re-running this cell replaces the
    # split instead of stacking a second one on top of it (which would leave the
    # same image in both training and testing)
    for folder in (TRAINING, TESTING):
        for old_file in os.listdir(folder):
            os.remove(os.path.join(folder, old_file))

    # copy files into their destination folders
    for filename in training_set:
        this_file = SOURCE + filename
        destination = TRAINING + filename
        copyfile(this_file, destination)

    for filename in testing_set:
        this_file = SOURCE + filename
        destination = TESTING + filename
        copyfile(this_file, destination)


CAT_SOURCE_DIR = "data/catsdogs/PetImages/Cat/"
TRAINING_CATS_DIR = "data/cats-v-dogs/training/cats/"
TESTING_CATS_DIR = "data/cats-v-dogs/testing/cats/"
DOG_SOURCE_DIR = "data/catsdogs/PetImages/Dog/"
TRAINING_DOGS_DIR = "data/cats-v-dogs/training/dogs/"
TESTING_DOGS_DIR = "data/cats-v-dogs/testing/dogs/"

split_size = .9
split_data(CAT_SOURCE_DIR, TRAINING_CATS_DIR, TESTING_CATS_DIR, split_size)
split_data(DOG_SOURCE_DIR, TRAINING_DOGS_DIR, TESTING_DOGS_DIR, split_size)

# Expected output (plus a few non-image files that are skipped)
# 666.jpg is zero length or not a valid image, so ignoring.
# 11702.jpg is zero length or not a valid image, so ignoring.


Thumbs.db is zero length or not a valid image, so ignoring.
666.jpg is zero length or not a valid image, so ignoring.


/Users/jaeyongjung/Desktop/coursera/cv_pytorch/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Thumbs.db is zero length or not a valid image, so ignoring.
11702.jpg is zero length or not a valid image, so ignoring.


Check that the training and test sets are the expected lengths.

In [6]:
# counts after the split
print("Number of training cat images", len(os.listdir('data/cats-v-dogs/training/cats/')))
print("Number of training dog images", len(os.listdir('data/cats-v-dogs/training/dogs/')))
print("Number of testing cat images", len(os.listdir('data/cats-v-dogs/testing/cats/')))
print("Number of testing dog images", len(os.listdir('data/cats-v-dogs/testing/dogs/')))

# expected output (approximately, a few corrupt files are dropped)
# Number of training cat images 11250
# Number of training dog images 11250
# Number of testing cat images 1250
# Number of testing dog images 1250


Number of training cat images 11249
Number of training dog images 11249
Number of testing cat images 1250
Number of testing dog images 1250


### Data augmentation (try adjusting the parameters)!

Here, you'll use torchvision `transforms` to perform data augmentation (the equivalent of Keras' `ImageDataGenerator`).
- Things like rotating and flipping the existing images allows you to generate training data that is more varied, and can help the model generalize better during training.  
- The validation set only gets the resizing and normalization.

You can use the default parameter values for a first pass through this lab.
- Later, try to experiment with the augmentation parameters to improve the model's performance.
- Try to drive reach 99.9% validation accuracy or better.

The pretrained InceptionV3 weights in torchvision were ported from TensorFlow and expect pixel values scaled to the range `[-1, 1]`, which is what `Normalize(mean=0.5, std=0.5)` does.

In [7]:
TRAINING_DIR = "data/cats-v-dogs/training/"

# per-image pipeline (no batch dim yet):
#   PIL image -> Resize             -> PIL, 150x150
#             -> RandomAffine       -> PIL, 150x150 (rotate/shift/shear/zoom)
#             -> RandomHorizontalFlip -> PIL, 150x150
#             -> ToTensor           -> shape (3, 150, 150), float in [0, 1], channels-first
#             -> Normalize          -> shape (3, 150, 150), float in ~[-1, 1]
# Experiment with your own parameters to reach 99.9% validation accuracy or better
train_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomAffine(degrees=40,              # rotation_range=40
                            translate=(0.2, 0.2),    # width/height_shift_range=0.2
                            shear=0.2,               # shear_range=0.2
                            scale=(0.8, 1.2)),       # zoom_range=0.2
    transforms.RandomHorizontalFlip(),               # horizontal_flip=True
    transforms.ToTensor(),                           # rescale=1./255
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
train_dataset = ImageFolder(TRAINING_DIR, transform=train_transform)
# shape: (3, 150, 150) samples -> (batch_size, 3, 150, 150) batches
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True, num_workers=4, persistent_workers=True)

VALIDATION_DIR = "data/cats-v-dogs/testing/"

# validation: resize + tensor + normalize only, no random augmentation
validation_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
validation_dataset = ImageFolder(VALIDATION_DIR, transform=validation_transform)
validation_loader = DataLoader(validation_dataset, batch_size=100, num_workers=4, persistent_workers=True)

print(train_dataset.class_to_idx)   # alphabetical folder order -> {cats: 0, dogs: 1}


{'cats': 0, 'dogs': 1}


### Get and prepare the model

You'll be using the `InceptionV3` model.  
- Since you're making use of transfer learning, you'll load the pre-trained weights of the model (torchvision downloads them automatically).
- You'll also freeze the existing layers so that they aren't trained on your downstream task with the cats and dogs data.
- You'll also cut the network at the layer `Mixed_6e` (this is what Keras calls `mixed7`) because you'll add some layers after this last layer.

In [8]:
# pretrained weights, downloaded automatically the first time
pre_trained_model = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)

# freeze pretrained weights: no gradients, no updates
for param in pre_trained_model.parameters():
    param.requires_grad = False

# keep layers up to 'Mixed_6e' (= Keras' 'mixed7'), drop everything after
layer_names = [name for name, _ in pre_trained_model.named_children()]
last_layer = 'Mixed_6e'
kept_layers = layer_names[:layer_names.index(last_layer) + 1]
feature_extractor = nn.Sequential(*[getattr(pre_trained_model, name) for name in kept_layers])

# shape in:  (1, 3, 150, 150)  batch=1, channels=3, height=150, width=150
# shape out: (1, C, H, W)      fewer H/W, more channels (learned features)
# find it empirically with a dummy all-zero batch, instead of computing it by hand
with torch.no_grad():
    last_output_shape = feature_extractor(torch.zeros(1, 3, 150, 150)).shape
print('last layer output shape: ', tuple(last_output_shape))


last layer output shape:  (1, 768, 7, 7)


### Add layers
Add some layers that you will train on the cats and dogs data.
- `Flatten`: This will take the output of the `last_layer` and flatten it to a vector.
- `Linear`: You'll add a fully connected layer with a relu activation.
- `Linear`: After that, add a fully connected layer with a single output. In Keras this layer had a sigmoid activation that scales the output to range from 0 to 1, and allows you to interpret the output as a prediction between two categories (cats or dogs). In PyTorch the sigmoid is folded into the loss (`nn.BCEWithLogitsLoss`) for numerical stability, and applied explicitly when you want a probability.

Then create the model object.

In [9]:
model = nn.Sequential(
    feature_extractor,
    nn.Flatten(),                                          # shape: (N, C, H, W) -> (N, C*H*W)
    # Linear applies one W/b to each sample independently -> in_features = C*H*W (skip batch dim N)
    nn.Linear(int(np.prod(last_output_shape[1:])), 1024),
    nn.ReLU(),
    nn.Linear(1024, 1),                                     # shape: (N, 1024) -> (N, 1) logit
).to(device)


### Train the model
Set up the loss and optimizer, and then train it on the data.
- Feel free to adjust the number of epochs.  This project was originally designed with 20 epochs.
- For the sake of time, you can use fewer epochs (2) to see how the code runs.

In [ ]:
model.parameters()

<generator object Module.parameters at 0x118809a80>

: 

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()   # sigmoid + binary cross-entropy on raw logits
# alpha/eps match Keras RMSprop defaults (rho=0.9, eps=1e-7); PyTorch's defaults are more aggressive
optimizer = torch.optim.RMSprop(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001, alpha=0.9, eps=1e-7)


def run_epoch(loader, model, loss_fn, optimizer, device, train, steps_per_epoch=None):
    '''
    Runs one pass over `loader`, training or evaluating.

    Args:
      loader (DataLoader) -- yields (images, labels) batches
      model (nn.Module) -- classifier being trained or evaluated
      loss_fn (callable) -- loss applied to (logits, labels)
      optimizer (torch.optim.Optimizer) -- updates weights; only used when train is True
      device (torch.device) -- device the batches are moved to
      train (bool) -- True updates the weights, False only measures
      steps_per_epoch (int) -- stop after this many batches, or None for the whole loader

    Returns:
      (float, float) -- mean loss and accuracy over the batches seen
    '''
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for step, (xb, yb) in enumerate(loader):
            if steps_per_epoch is not None and step >= steps_per_epoch:
                break
            # shape: xb (N, 3, 150, 150) | yb (N,) -> (N, 1) float, to match logits
            # unsqueeze(1) inserts a size-1 axis at position 1; squeeze is its inverse
            xb, yb = xb.to(device), yb.to(device).float().unsqueeze(1)
            logits = model(xb)   # shape: (N, 1)
            loss = loss_fn(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            correct += ((logits > 0) == (yb > 0.5)).sum().item()   # logits > 0 <=> sigmoid > 0.5
            count += len(xb)
    return total_loss / count, correct / count


EPOCHS = 2**3   # bump this up for better accuracy
history = {'acc': [], 'val_acc': [], 'loss': [], 'val_loss': []}
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, model, loss_fn, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(validation_loader, model, loss_fn, optimizer, device, train=False)
    history['loss'].append(train_loss); history['acc'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")


/Users/jaeyongjung/Desktop/coursera/cv_pytorch/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1/32 - loss: 0.3566 - acc: 0.8330 - val_loss: 0.2272 - val_acc: 0.9092


/Users/jaeyongjung/Desktop/coursera/cv_pytorch/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 2/32 - loss: 0.2828 - acc: 0.8755 - val_loss: 0.1699 - val_acc: 0.9332


/Users/jaeyongjung/Desktop/coursera/cv_pytorch/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 3/32 - loss: 0.2601 - acc: 0.8868 - val_loss: 0.2070 - val_acc: 0.9244
Epoch 4/32 - loss: 0.2476 - acc: 0.8928 - val_loss: 0.1705 - val_acc: 0.9364
Epoch 5/32 - loss: 0.2428 - acc: 0.8948 - val_loss: 0.1621 - val_acc: 0.9372


/Users/jaeyongjung/Desktop/coursera/cv_pytorch/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 6/32 - loss: 0.2381 - acc: 0.8973 - val_loss: 0.1782 - val_acc: 0.9356


### Visualize the training and validation accuracy

You can see how the training and validation accuracy change with each epoch on an x-y plot.

In [ ]:
%matplotlib inline

import matplotlib.image  as mpimg
import matplotlib.pyplot as plt

# one value per epoch, filled in during training above
acc=history['acc']
val_acc=history['val_acc']
loss=history['loss']
val_loss=history['val_loss']

epochs=range(len(acc))

plt.plot(epochs, acc, 'r', label="Training Accuracy")
plt.plot(epochs, val_acc, 'b', label="Validation Accuracy")
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()


### Predict on a test image

You can use any image and have the model predict whether it's a dog or a cat.
- Find an image of a dog or cat and put its path in the `image_paths` list below (a couple of sample images are downloaded for you).
- Run the following code cell.
- The model will print "is a dog" or "is a cat" depending on the model's prediction.

In [ ]:
sample_images = {
    "data/cat1.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/cat1.jpeg",
    "data/dog1.jpg": "https://storage.googleapis.com/tensorflow-1-public/tensorflow-3-temp/MLColabImages/dog1.jpeg",
}
for path, url in sample_images.items():
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

image_paths = list(sample_images.keys())

model.eval()
for fn in image_paths:

  # predicting images
  img = Image.open(fn).convert("RGB")     # force 3 channels
  x = validation_transform(img)          # shape: (3, 150, 150)
  x = x.unsqueeze(0).to(device)          # add batch dim -> shape: (1, 3, 150, 150)

  with torch.no_grad():
    classes = torch.sigmoid(model(x)).cpu().numpy()   # (1, 1) logit -> probability
  print(classes[0])
  if classes[0]>0.5:
    print(fn + " is a dog")
  else:
    print(fn + " is a cat")
